# End-to-end simulation of observation and pipeline reduction of Barnard's star as a prevision RV standard

<span style="color:red">The HISPEC DRP is in active development. Please submit issues and suggestions on the [github](https://github.com/oirlab/HISPEC_DRP).</span>


This notebook shows two examples of simulating data:

1. A simulation of a single up-the-ramp exposure of a star with a theoretical spectrum with similar properties to Bernard's star, assuming no planets.
2. A segment of a precision radial velocity observations of the same star, now with an orbiting super Earth causing the radial velocity to vary.

For each exposure, the notebook simulates a BSPEC rate map from a Phoenix template (approximately similar to Barnard's star), builds a `RampModel`, then runs `Stage0Pipeline` → `Stage1Pipeline` → `Stage2Pipeline`.

---

## Setup

**Environment variables** (set these in your shell / notebook kernel before running):

| Variable | Required when | Purpose |
|---|---|---|
| `KOA_CALIBRATION_CACHE` | Always | Local KOA calibration cache directory (and SQLite DB) |
| `KECK_OBSERVER_EMAIL` | Syncing / downloading from KOA | Keck Observer account email |
| `KECK_OBSERVER_PASSWORD` | Syncing / downloading from KOA | Keck Observer account password |

If `KOA_CALIBRATION_CACHE` is unset, the setup cell sets it under `OUT`.  
The KOA cell below syncs metadata from remote KOA and downloads any **BSPEC** calibration files that are not already in the cache.


In [ ]:
import os
import hispecdrp

print(f"hispecdrp {hispecdrp.__version__} @ {hispecdrp.__file__}")

# OUTPUT DIRECTORY — update this to your desired output directory
OUT = "/path/to/barnards_star_prv_example"
os.makedirs(OUT, exist_ok=True)

os.environ.setdefault(
    "KOA_CALIBRATION_CACHE",
    os.path.join(OUT, "cals", "KOA_CALIBRATION_CACHE"),
)
print("output:", OUT)
print("KOA_CALIBRATION_CACHE:", os.environ["KOA_CALIBRATION_CACHE"])


## 1. Sync / verify KOA caldb (BSPEC)


In [ ]:
from hispecdrp.calibrations import verify_koa_calibrations

# Requires env vars: KOA_CALIBRATION_CACHE, KECK_OBSERVER_EMAIL, KECK_OBSERVER_PASSWORD
SPECTROGRAPH = "BSPEC"
summary = verify_koa_calibrations(SPECTROGRAPH, ignore_missing_cal_types=["lfc_wavesol"])

print(
    f"{summary['spectrograph']} caldb: {summary['n_records']} record(s); "
    f"downloaded {summary['n_downloaded']}/{summary['n_missing']} missing file(s)"
)
print("types:", summary["cal_types"])
if summary["downloaded"]:
    print("downloaded:", summary["downloaded"])
if summary["failed"]:
    print("failed:", summary["failed"])


## 2. Single exposure of star with no planet

Stellar properties are approximately similar to Bernard's star (same Teff, logg, and metallicity).

### 2.1 Simulated rate map from a 1D spectrum → `RampModel`

Load a Phoenix template, densify it, Doppler-shift the densified 1D spectrum by
`radial_velocity_km_s` (`inject_rv_via='spectrum'`, default), paint with ZEMAX, then
build a raw UTR `RampModel`. Stage-0 KOA masters plus sim Stage-1 products
(packaged line-ZP WAVE — **unscaled**, same bake for every RV — traces, fiberflat)
are registered into the local caldb.

1. Fiber 1 is the on the star assuming ideal coupling. 
2. Fiber 2 is empty (will later be updated to include etalon spectrum). 
3. Fiber 3 is background sky flux.

What this simulation does **not** include:
- telluric injection/correction
- wavelength solution determination (here the wave solution is derived from a ZEMAX instrument model)
- atmospheric variability of any kind

In [ ]:
import numpy as np
import astropy.units as u

# read etalon spectrum
etalon_wvs, etalon_trans = np.loadtxt('/home/agibbs/Downloads/etalon_1048A_line_list.csv', delimiter=',', usecols=[1,4], skiprows=1).T
etalon_wvs = (etalon_wvs*u.nm).to(u.micron)
etalon_flux = 10000*u.ph/u.s * etalon_trans


In [ ]:
from hispecdrp.simulations.simulate_up_the_ramp import create_ramp_from_spectrum

FIBERS = [1,2,3]
READ_TIME = 1.7 # seconds


# Injected stellar RV (km/s). Default path densifies the 1D template, Doppler-shifts
# it, paints with ZEMAX, and attaches the fixed packaged line-ZP WAVE (no WAVE×(1+z)).
STELLAR_RV = -110.1

exposure_time = 60 # seconds
n_reads = int(exposure_time/READ_TIME)

# Create a UTR map from a controlled seed and template spectrum
# The user can normalize to a specific magnitude (AB mag) and 2MASS filter
ramp, cals, rate_map = create_ramp_from_spectrum(
    OUT,
    spectrograph=SPECTROGRAPH,
    fibers=FIBERS,
    template_filename={
        1:"lte03200-5.00-0.5.PHOENIX-ACES-AGSS-COND-2011-HiRe.h5",
    },
    mag_ab=6.15,
    filter_band="J",  # BSPEC (yJ)
    fiber_spectra={
        2:(etalon_wvs,etalon_flux),
    },
    n_reads=n_reads,
    seed=0,
    radial_velocity_km_s={
        1:STELLAR_RV,
    },
    barycentric_velocity_km_s=0.0,
    inject_rv_via="spectrum",
    sky_filename='mk_skybg_zm_1.0_1.0_ph.dat',
    apply_sky=[1,3],
    apply_e2e_throughput=[1,3]
)

print(f"fibers: {FIBERS}")
print(f"ramp type: {ramp.meta.model_type}")
print(f"ramp filename: {ramp.meta.filename}")
print(f"ramp data shape: {ramp.data.shape}")
print(f"Stage-0 cals: {sorted(cals)}")
print(f"rate_map shape: {rate_map.shape}")
print(f"wavesol note: {getattr(getattr(cals.get('wavesol'), 'meta', None), 'note', None)}")


### 2.2 Stage 0 → 1 → 2 via pipeline classes


In [ ]:
from hispecdrp.pipelines import Stage0Pipeline, Stage1Pipeline, Stage2Pipeline

l0 = Stage0Pipeline(output_dir=OUT).apply(ramp)

do_fibers = [str(f) for f in FIBERS]
# Fixed packaged line-ZP WAVE from create_ramp_from_spectrum (unscaled for spectrum paint)
stage1_args = {
    "spec_extract": {"do_fibers": do_fibers},
    "set_wavesol": {"wavesol": cals["wavesol"]},
}
l1 = Stage1Pipeline(output_dir=OUT, prim_args=stage1_args).apply(l0)

injected_m_s = float(STELLAR_RV) * 1000.0
l2 = Stage2Pipeline(
    output_dir=OUT,
    prim_args={
        "blaze_corr": {"do_fibers": do_fibers},
        "stellar_rv": {
            "ccf_input_velocity": injected_m_s,
            "do_fibers": [1],
            "save_preview": True,
            "preview_write_png": False,
        },
        "sky_sub": {"run": True},
    },
).apply(l1)

print(f"L0 type: {l0.meta.model_type}")
print(f"L0 filename: {l0.meta.filename}")
print(f"L1 type: {l1.meta.model_type}")
print(f"L1 filename: {l1.meta.filename}")
print(f"L2 type: {l2.meta.model_type}")
print(f"L2 filename: {l2.meta.filename}")

## 3. Segment of PRV observations

We now simulate multiple exposures over several nights for a super Earth planet orbiting the same Barnard's star analog.

### 3.1 Simulating rate maps from an initial 1D spectrum and varying RV

1. Fiber 1 is the on the star assuming ideal coupling. 
2. Fiber 2 is empty (will later be updated to include etalon spectrum). 
3. Fiber 3 is background sky flux.

What this simulation does **not** include:
- telluric injection/correction
- wavelength solution determination (here the wave solution is derived from a ZEMAX instrument model)
- atmospheric variability of any kind
- stellar variability of any kind

In [ ]:
import numpy as np


FIBERS = [1,3]
READ_TIME = 1.7 # seconds

# Injected stellar RV (km/s).
STELLAR_RV = -110.1

# observation parameters
starting_jd = 2461293.91667 # sep 10 2026
nights = 3
exposures_per_night = 3
exposure_time = 60 # seconds
n_reads = int(exposure_time/READ_TIME)

nightly_start_jd_arr = starting_jd + np.arange(0,nights)
time_offsets = np.linspace(0, (exposure_time+READ_TIME)/86400 * (exposures_per_night - 1), exposures_per_night)
obs_jd_arr = np.add.outer(nightly_start_jd_arr, time_offsets).flatten()

# planet parameters
phase0_jd = starting_jd 
semiamplitude_K = 15 # m/s
period = 1.75 # days

phase_arr = (obs_jd_arr - phase0_jd) / period
rv_arr = semiamplitude_K * np.sin(2 * np.pi * phase_arr)/1000 + STELLAR_RV


Define function to simulate and get L2 data product from each exposure

In [ ]:
from hispecdrp.simulations.simulate_up_the_ramp import create_ramp_from_spectrum
from hispecdrp.pipelines import Stage0Pipeline, Stage1Pipeline, Stage2Pipeline
import gc

def run_one_injection(label: str, injected_rv_km_s: float, mjd_start: float, seed: int) -> dict:
    """Build one UTR, reduce Stage0–2, return L2 data product."""
    run_dir = OUT
    os.makedirs(run_dir, exist_ok=True)
    injected_m_s = float(injected_rv_km_s) * 1000.0

    print(f"\n=== {label}: inject {injected_rv_km_s:.6f} km/s, seed={seed} ===")
    ramp, cals, rate_map = create_ramp_from_spectrum(
        run_dir,
        spectrograph=SPECTROGRAPH,
        fibers=FIBERS,
        template_filename="lte03200-5.00-0.5.PHOENIX-ACES-AGSS-COND-2011-HiRe.h5",
        mag_ab=6.15,
        filter_band="J",  # BSPEC (yJ)
        n_reads=n_reads,
        seed=seed,
        radial_velocity_km_s=injected_rv_km_s,
        barycentric_velocity_km_s=0.0,
        inject_rv_via='spectrum',
        sky_filename='mk_skybg_zm_1.0_1.0_ph.dat',
        sky_fiber=3,
        apply_e2e_throughput=FIBERS,
        mjd_start=mjd_start,
        register_stage1_sims=False,
        register_wavesol=False,
        save_ramp=False,
    )
    wmeta = getattr(cals.get("wavesol"), "meta", None)
    print(
        f"ramp={ramp.meta.filename}  shape={ramp.data.shape}  "
        f"wavesol note={getattr(wmeta, 'note', None)}  "
        f"injected_rv_via={getattr(wmeta, 'injected_rv_via', None)}"
    )

    do_fibers = [str(f) for f in FIBERS]
    l0 = Stage0Pipeline(output_dir=run_dir, save_result=False).apply(ramp)

    del ramp
    del rate_map

    l1 = Stage1Pipeline(
        output_dir=run_dir,
        save_result=False,
        prim_args={
            "spec_extract": {"do_fibers": do_fibers},
            "set_wavesol": {"wavesol": cals["wavesol"]},
        },
    ).apply(l0)

    del l0

    l2 = Stage2Pipeline(
        output_dir=run_dir,
        save_result=True,
        prim_args={
            "blaze_corr": {"do_fibers": do_fibers},
            "stellar_rv": {
                "ccf_input_velocity": injected_m_s,
                "do_fibers": list(FIBERS),
                "save_preview": False,
                "preview_write_png": False,
            },
            "sky_sub": {"run": False},
        },
    ).apply(l1)

    del l1
    gc.collect()

    return {
        "label": label,
        "mjd": mjd_start,
        "run_dir": run_dir,
        "seed": seed,
        "inject_km_s": float(injected_rv_km_s),
        "inject_m_s": injected_m_s,
        "wavesol": cals["wavesol"],
        "l2": l2,
    }


Now run

In [ ]:
l2_results = {}

for i in range(nights*exposures_per_night):
    l2_results[str(obs_jd_arr[i])] = run_one_injection(label=str(obs_jd_arr[i]),
                                                       injected_rv_km_s=rv_arr[i],
                                                       mjd_start=obs_jd_arr[i],
                                                       seed=0)